In [1]:
import pandas as pd
import numpy as np

In [2]:
df = pd.read_csv("bank-full.csv", sep=";")
df

,age,job,marital,education,default,balance,housing,loan,contact,day,month,duration,campaign,pdays,previous,poutcome,y
0,58,management,married,tertiary,no,2143,yes,no,unknown,5,may,261,1,-1,0,unknown,no
1,44,technician,single,secondary,no,29,yes,no,unknown,5,may,151,1,-1,0,unknown,no
2,33,entrepreneur,married,secondary,no,2,yes,yes,unknown,5,may,76,1,-1,0,unknown,no
3,47,blue-collar,married,unknown,no,1506,yes,no,unknown,5,may,92,1,-1,0,unknown,no
4,33,unknown,single,unknown,no,1,no,no,unknown,5,may,198,1,-1,0,unknown,no
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
45206,51,technician,married,tertiary,no,825,no,no,cellular,17,nov,977,3,-1,0,unknown,yes
45207,71,retired,divorced,primary,no,1729,no,no,cellular,17,nov,456,2,-1,0,unknown,yes
45208,72,retired,married,secondary,no,5715,no,no,cellular,17,nov,1127,5,184,3,success,yes
45209,57,blue-collar,married,secondary,no,668,no,no,telephone,17,nov,508,4,-1,0,unknown,no


In [3]:
df = df[["age", "job", "marital", "education", "balance", "housing", "contact", "day", "month", "duration", "campaign", "pdays", "previous", "poutcome", "y"]]
df

,age,job,marital,education,balance,housing,contact,day,month,duration,campaign,pdays,previous,poutcome,y
0,58,management,married,tertiary,2143,yes,unknown,5,may,261,1,-1,0,unknown,no
1,44,technician,single,secondary,29,yes,unknown,5,may,151,1,-1,0,unknown,no
2,33,entrepreneur,married,secondary,2,yes,unknown,5,may,76,1,-1,0,unknown,no
3,47,blue-collar,married,unknown,1506,yes,unknown,5,may,92,1,-1,0,unknown,no
4,33,unknown,single,unknown,1,no,unknown,5,may,198,1,-1,0,unknown,no
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
45206,51,technician,married,tertiary,825,no,cellular,17,nov,977,3,-1,0,unknown,yes
45207,71,retired,divorced,primary,1729,no,cellular,17,nov,456,2,-1,0,unknown,yes
45208,72,retired,married,secondary,5715,no,cellular,17,nov,1127,5,184,3,success,yes
45209,57,blue-collar,married,secondary,668,no,telephone,17,nov,508,4,-1,0,unknown,no


In [4]:
df.isnull().sum()

age          0
job          0
marital      0
education    0
balance      0
housing      0
contact      0
day          0
month        0
duration     0
campaign     0
pdays        0
previous     0
poutcome     0
y            0
dtype: int64

# Question 1

In [5]:
df.education.mode()

0    secondary
Name: education, dtype: object

# Question 2

In [6]:
df[["age"]].corrwith(df.balance)

age    0.097783
dtype: float64

In [7]:
df[["campaign", "pdays"]].corrwith(df.day)

campaign    0.162490
pdays      -0.093044
dtype: float64

In [8]:
df[["pdays"]].corrwith(df.previous)

pdays    0.45482
dtype: float64

In [9]:
df.y

0         no
1         no
2         no
3         no
4         no
        ... 
45206    yes
45207    yes
45208    yes
45209     no
45210     no
Name: y, Length: 45211, dtype: object

In [10]:
df = df.copy()
df['y'] = df.y.map({'yes': 1, 'no': 0})
df.y

0        0
1        0
2        0
3        0
4        0
        ..
45206    1
45207    1
45208    1
45209    0
45210    0
Name: y, Length: 45211, dtype: int64

In [11]:
from sklearn.model_selection import train_test_split

In [12]:
df_full_train, df_test = train_test_split(df, test_size=0.2, random_state=42)
df_train, df_val = train_test_split(df_full_train, test_size=0.25, random_state=42)
len(df_train), len(df_val), len(df_test)

(27126, 9042, 9043)

In [13]:
df_train = df_train.reset_index(drop=True)
df_val = df_val.reset_index(drop=True)
df_test = df_test.reset_index(drop=True)

y_train = df_train.y
y_val = df_val.y
y_test = df_test.y

In [14]:
del df_train['y']
del df_val['y']
del df_test['y']

# Question 3

In [15]:
categorical = ["job", "marital", "education", "housing", "loan", "contact", "month", "poutcome"]

In [52]:
numerical = ["age", "balance", "day", "duration", "campaign", "pdays", "previous"]

In [16]:
from sklearn.metrics import mutual_info_score

In [17]:
def mutual_info_y_score(series):
    return mutual_info_score(series, y_train)

In [18]:
score = df_train[categorical].apply(mutual_info_y_score)
round(score,2)

job          0.01
marital      0.00
education    0.00
housing      0.01
contact      0.01
month        0.03
poutcome     0.03
dtype: float64

# Question 4

In [19]:
from sklearn.feature_extraction import DictVectorizer

In [26]:
train_dict = df_train.to_dict(orient='records')
train_dict[0]

{'age': 32,
 'job': 'technician',
 'marital': 'single',
 'education': 'tertiary',
 'balance': 1100,
 'housing': 'yes',
 'contact': 'cellular',
 'day': 11,
 'month': 'aug',
 'duration': 67,
 'campaign': 1,
 'pdays': -1,
 'previous': 0,
 'poutcome': 'unknown'}

In [21]:
dv = DictVectorizer(sparse=False)
X_train = dv.fit_transform(train_dict)
dv.get_feature_names_out()

array(['age', 'balance', 'campaign', 'contact=cellular',
       'contact=telephone', 'contact=unknown', 'day', 'duration',
       'education=primary', 'education=secondary', 'education=tertiary',
       'education=unknown', 'housing=no', 'housing=yes', 'job=admin.',
       'job=blue-collar', 'job=entrepreneur', 'job=housemaid',
       'job=management', 'job=retired', 'job=self-employed',
       'job=services', 'job=student', 'job=technician', 'job=unemployed',
       'job=unknown', 'marital=divorced', 'marital=married',
       'marital=single', 'month=apr', 'month=aug', 'month=dec',
       'month=feb', 'month=jan', 'month=jul', 'month=jun', 'month=mar',
       'month=may', 'month=nov', 'month=oct', 'month=sep', 'pdays',
       'poutcome=failure', 'poutcome=other', 'poutcome=success',
       'poutcome=unknown', 'previous'], dtype=object)

In [44]:
val_dict = df_val.to_dict(orient='records')
X_val = dv.fit_transform(val_dict)

In [23]:
from sklearn.linear_model import LogisticRegression

In [24]:
y_train.unique

<bound method Series.unique of 0        0
1        0
2        0
3        0
4        0
        ..
27121    0
27122    0
27123    0
27124    1
27125    0
Name: y, Length: 27126, dtype: int64>

In [25]:
model = LogisticRegression(solver='liblinear', C=1.0, max_iter=1000, random_state=42)
model.fit(X_train, y_train)

LogisticRegression(max_iter=1000, random_state=42, solver='liblinear')

In [45]:
y_pred = model.predict_proba(X_val)[:,1]

In [46]:
actual_pred = (y_pred >= 0.5)

In [63]:
original = (y_val == actual_pred).mean()

# Question 5

In [49]:
def generate_X(df_train, df_val, features):
    train_dict = df_train[features].to_dict(orient='records')
    val_dict = df_val[features].to_dict(orient='records')
    dt = DictVectorizer(sparse=False)
    dv = DictVectorizer(sparse=False)
    return dt.fit_transform(train_dict), dv.fit_transform(val_dict)

In [71]:
def predict(X_train, X_val, y_train, y_val, C=1.0):
    model = LogisticRegression(solver='liblinear', C=C, max_iter=1000, random_state=42)
    model.fit(X_train, y_train)
    y_pred = model.predict_proba(X_val)[:, 1]
    actual_pred = (y_pred >= 0.5)
    return (y_val == actual_pred).mean()

In [69]:
def aggregate(df_train, df_val, y_train, y_val, features):
    prediction_without = dict()
    for feature in features:
        train = df_train.copy()
        val = df_val.copy()
        del train[feature]
        del val[feature]
        without = features.copy()
        without.remove(feature)
        X_train, X_val = generate_X(train, val, without)
        prediction_without[feature] = abs(original - predict(X_train, X_val, y_train, y_val))
    return prediction_without

In [70]:
pred = aggregate(df_train, df_val, y_train, y_val, categorical + numerical)
pred

{'job': np.float64(0.0002211900022119906),
 'marital': np.float64(0.0),
 'education': np.float64(0.0),
 'housing': np.float64(0.0002211900022119906),
 'contact': np.float64(0.00044238000442375913),
 'month': np.float64(0.0011059500110593978),
 'poutcome': np.float64(0.007520460075204571),
 'age': np.float64(0.00044238000442387015),
 'balance': np.float64(0.0001105950011059953),
 'day': np.float64(0.00044238000442387015),
 'duration': np.float64(0.011170095111700862),
 'campaign': np.float64(0.0006635700066356387),
 'pdays': np.float64(0.0),
 'previous': np.float64(0.0)}

# Question 6

> tested with Ridge as it was stated to do a regularized logistic regression but the solver parameter was not the same and it didn't contain the predict_proba method.

In [88]:
def predict_reg(X_train, X_val, y_train, y_val, C=1.0):
    model = LogisticRegression(solver='liblinear', C=C, max_iter=1000, random_state=42)
    model.fit(X_train, y_train)
    y_pred = model.predict_proba(X_val)[:, 1]
    actual_pred = (y_pred >= 0.5)
    return (y_val == actual_pred).mean()

In [89]:
for c in (0.01, 0.1, 1, 10, 100):
    X_train, X_val = generate_X(df_train, df_val, categorical + numerical)
    print(f"{c}: {round(predict_reg(X_train, X_val, y_train, y_val, C=c), 3)}")

0.01: 0.898
0.1: 0.901
1: 0.901
10: 0.901
100: 0.901
